# Summary
This notebook is an ongoing summary of the content covered in the course. It primarily follows a series of steps.

## 1. Load Data

In [16]:
import pandas as pd

adult_census = pd.read_csv("1 Predictive Modeling Pipeline/adult.csv")

In [17]:
target_name = "class"
target = adult_census[target_name]

In [18]:
exclude_features = [target_name, "education-num"]
data = adult_census.drop(columns=exclude_features)

## 2. Explore the Data
While the exact code is not detailed here, this involves exploring the features and target, their distributions, other information, etc. This can include visualizations like histograms.

In [19]:
adult_census.dtypes

id                int64
age               int64
workclass           str
fnlwgt            int64
education           str
education-num     int64
marital-status      str
occupation          str
relationship        str
race                str
sex                 str
capital-gain      int64
capital-loss      int64
hours-per-week    int64
native-country      str
class               str
dtype: object

In [20]:
adult_census[target_name].value_counts()

class
<=50K    37155
>50K     11687
Name: count, dtype: int64

In [21]:
data.shape

(48842, 14)

## 3. Train-Test Data Split

In [22]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    data, target, test_size=0.2, random_state=42
)

## 4. Create Model Pipeline
An object that has a `fit` method is called an **estimator**. `fit` takes some input data and transforms the *model states* using some algorithm. There are two common subclasses of estimators:
- **Predictors**: objects with a `predict` method. These are the machine learning models, like KNN or Logistic Regressions. `fit` will use a learning algorithm to set the model state; `predict` will use the model states with some data to make predictions.
- **Transformers**: objects with a `transform` method. These are things like scalars and encoders. `fit` will adjust the model states; `transform` will make changes to the data.

### a. Select Column Types
Separate the column by data types - categorical or numerical.

In [23]:
from sklearn.compose import make_column_selector as selector

numerical_selector = selector(dtype_exclude=object)
categorical_selector = selector(dtype_include=object)

numerical_columns = numerical_selector(data)
categorical_columns = categorical_selector(data)

### b. Dispatch to Preprocessors
Create preprocessors for the data types.

In [24]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_preprocessor = OneHotEncoder(handle_unknown="ignore")
numerical_preprocessor = StandardScaler()

### c. Apply Transformer
Combine the transformers into a single `ColumnTransformer` to automatically handle the work.

In [25]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ("encoder", categorical_preprocessor, categorical_columns),
        ("scaler", numerical_preprocessor, numerical_columns),
    ],
    remainder="passthrough"
)

### d. Create Pipeline
Pipelines will sequentially call their underlying transformers and predictors in one place.

In [40]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            random_state=42
        )),
    ]
)

### e. Fit the Model
Fit the model, and test the generalization accuracy with `score` on the testing data.

In [41]:
model.fit(
    X=X_train,
    y=y_train
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('encoder', ...), ('scaler', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different tra

In [37]:
accuracy = model.score(
    X=X_test,
    y=y_test
)
accuracy

Parameter memory: None
Parameter steps: [('preprocessor', ColumnTransformer(transformers=[('encoder',
                                 OneHotEncoder(handle_unknown='ignore'),
                                 ['workclass', 'education', 'marital-status',
                                  'occupation', 'relationship', 'race', 'sex',
                                  'native-country']),
                                ('scaler', StandardScaler(),
                                 ['id', 'age', 'fnlwgt', 'capital-gain',
                                  'capital-loss', 'hours-per-week'])])), ('classifier', LogisticRegression(random_state=42))]
Parameter transform_input: None
Parameter verbose: False
Parameter preprocessor: ColumnTransformer(transformers=[('encoder',
                                 OneHotEncoder(handle_unknown='ignore'),
                                 ['workclass', 'education', 'marital-status',
                                  'occupation', 'relationship', 'race', 'sex',

## 5. Evaluate with Cross-Validation

In [58]:
from sklearn.model_selection import cross_validate

cv_result = cross_validate(
    model,
    data,
    target
)
cv_result

{'fit_time': array([0.20195222, 0.23194003, 0.20796704, 0.22044277, 0.21063995]),
 'score_time': array([0.02147079, 0.02156806, 0.02086496, 0.02233505, 0.02099204]),
 'test_score': array([0.85894155, 0.86078411, 0.86036036, 0.86312449, 0.84009009])}

## 6. Tune Hyperparameters
Using either grid-search or randomized-search, we can specify the different values of the hyperparameters we would like to tune.

When `fit` is called, the model embedded in the search is trained with every combination of hyperparameter. The best combination is selected by keeping the combination leading to the best mean cross-validated score.

In [35]:
from scipy.stats import loguniform

class loguniform_int:
    """Integer valued version of the log-uniform distribution"""

    def __init__(self, a, b):
        self._distribution = loguniform(a, b)

    def rvs(self, *args, **kwargs):
        """Random variable sample"""
        return self._distribution.rvs(*args, **kwargs).astype(int)

In [45]:
from sklearn.preprocessing import OrdinalEncoder

categorical_selector = selector(dtype_include=object)
categorical_columns = categorical_selector(data)

categorical_preprocessor = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical_preprocessor", categorical_preprocessor, categorical_columns)
    ],
    remainder="passthrough"
)

In [49]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", HistGradientBoostingClassifier(
            random_state=42,
            max_leaf_nodes=4
        ))
    ]
)

In [50]:
from sklearn.model_selection import RandomizedSearchCV

param_distributions = {
    "classifier__l2_regularization": loguniform(1e-6, 1e3),
    "classifier__learning_rate": loguniform(0.001, 10),
    "classifier__max_leaf_nodes": loguniform_int(2, 256),
    "classifier__min_samples_leaf": loguniform_int(1, 100),
    "classifier__max_bins": loguniform_int(2, 255),
}

model_random_search = RandomizedSearchCV(
    model,
    param_distributions=param_distributions,
    n_iter=10,
    cv=5,
    verbose=1,
)

In [51]:
model_random_search.fit(
    X=X_train,
    y=y_train
)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'classifier__l2_regularization': <scipy.stats....t 0x113dba120>, 'classifier__learning_rate': <scipy.stats....t 0x113dba210>, 'classifier__max_bins': <__main__.log...t 0x114293750>, 'classifier__max_leaf_nodes': <__main__.log...t 0x113782470>, ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will 

In [53]:
model_random_search.best_params_

{'classifier__l2_regularization': np.float64(0.16925795027788687),
 'classifier__learning_rate': np.float64(0.262211993093692),
 'classifier__max_bins': np.int64(20),
 'classifier__max_leaf_nodes': np.int64(4),
 'classifier__min_samples_leaf': np.int64(1)}

In [52]:
column_results = [f"param_{name}" for name in param_distributions.keys()]
column_results += ["mean_test_score", "std_test_score", "rank_test_score"]

cv_results = pd.DataFrame(model_random_search.cv_results_)
cv_results = cv_results[column_results].sort_values(
    "mean_test_score", ascending=False
)


def shorten_param(param_name):
    if "__" in param_name:
        return param_name.rsplit("__", 1)[1]
    return param_name


cv_results = cv_results.rename(shorten_param, axis=1)
cv_results

,l2_regularization,learning_rate,max_leaf_nodes,min_samples_leaf,max_bins,mean_test_score,std_test_score,rank_test_score
5,0.169258,0.262212,4,1,20,0.856167,0.003286,1
4,6.741083,0.043448,4,1,31,0.838789,0.002141,2
1,0.000001,0.063272,61,11,2,0.803701,0.001697,3
7,0.000004,0.022711,7,44,2,0.796458,0.002585,4
0,940.739013,0.004868,118,1,2,0.759501,0.000054,5
2,0.000074,0.001594,10,30,31,0.759501,0.000054,5
9,0.671056,0.002597,102,2,63,0.759501,0.000054,5
6,641.018852,4.624299,2,4,155,0.717656,0.005340,8
8,0.002313,2.097867,58,2,13,0.656082,0.130800,9
3,0.002602,5.689888,177,24,13,0.602330,0.075235,10


In [54]:
accuracy = model_random_search.score(
    X=X_test,
    y=y_test
)
accuracy

0.8637526870713481

When performing hyperparameter tuning, each combination of parameters will perform cross-validation. However, this is only done on the training dataset.

When we receive our model, fit with the best hyperparameters, it is best practice to perform another cross-validation (nested cross-validation), which includes the testing dataset.

In [59]:
cv_results = cross_validate(
    model_random_search,
    data,
    target,
    cv=5,
    n_jobs=2,
    return_train_score=True
)
cv_results

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Fitting 5 folds for each of 10 candidates, totalling 50 fits


/Users/michael/Development/Learning/scikit/.venv/lib/python3.14/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Fitting 5 folds for each of 10 candidates, totalling 50 fits
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Fitting 5 folds for each of 10 candidates, totalling 50 fits


{'fit_time': array([28.49328732, 29.25324416, 24.14018798, 31.06043482, 12.76820993]),
 'score_time': array([0.03470588, 0.03165817, 0.03249002, 0.04031801, 0.01961112]),
 'test_score': array([0.85597298, 0.86231958, 0.85677723, 0.87325962, 0.85933661]),
 'train_score': array([0.87431218, 0.86555934, 0.86323386, 0.87989456, 0.85727082])}